In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import re
import scipy.io
import psutil
import os
import math
import cmath


infile_path = 'Doherty_scaled.csv'
df = pd.read_csv(infile_path)

xin_real = df['xI_real']
xin_imag = df['xI_imag']
yout_real = df['yout_real']
yout_imag = df['yout_imag']

t_ns = np.arange(1,120001)

# read real, imaginary parts and calculate amplitude
xin = xin_real + 1j*xin_imag
xin_amp = abs(xin)
#xin_phase = phase(xin)
xinn = xin/max(xin_amp)
xinn_amp = abs(xinn)

yout = yout_real + 1j*yout_imag
yout_amp = abs(yout)
# normalize output to peak value
youtn = yout/max(yout_amp)
youtn_amp = abs(youtn)
#yout_phase = cmath.phase(y_out)

# compute rms value for each signal
xin_rms = np.sqrt(np.mean(np.abs(xin)**2))
yout_rms = np.sqrt(np.mean(np.abs(yout)**2))
gainrms_dB = 20*math.log10(yout_rms/xin_rms)
print(gainrms_dB)

y_arr = np.column_stack((xin_real, xin_imag))
youtn = np.array(youtn)
youtn_real = youtn.real
youtn_imag = youtn.imag
X_arr = np.column_stack((youtn_real, youtn_imag))

X = pd.DataFrame(X_arr, columns=['yout_real', 'yout_imag'])
y = pd.DataFrame(y_arr, columns=['xin_real', 'xin_imag'])


X_new = X.iloc[90000:120001]
print("Shape of X_new:", X_new.shape)
# Keep the first 90,000 rows in both dataframes:
X = X.iloc[:90000]
y = y.iloc[:90000]

print("Shape of y:", y.shape)

11.096999415866108
Shape of X_new: (30001, 2)
Shape of y: (90000, 2)


In [2]:
def make_time_delays(X_df: pd.DataFrame, y_df: pd.DataFrame, n_delay: int):
    X_arr = X_df.values            # shape (N, 2)
    y_arr = y_df.values            # shape (N, 2)
    N = len(X_arr)
    
    X_td = []
    y_td = []
    # for each t >= n_delay, stack [X[t-n_delay], ..., X[t]] ? window length = (n_delay+1)*2
    for t in range(n_delay, N):
        window = X_arr[t-n_delay:t+1].ravel()
        X_td.append(window)
        y_td.append(y_arr[t])
    
    return np.vstack(X_td), np.vstack(y_td)

def make_time_delays_array(X_arr, n_delay):
    """
    Given X_arr shape (N, 2), returns windows shape (N-n_delay, (n_delay+1)*2)
    """
    windows = []
    for t in range(n_delay, X_arr.shape[0]):
        window = X_arr[t-n_delay:t+1].ravel()
        windows.append(window)
    return np.vstack(windows)

n_delay = 0
X_td, y_td = make_time_delays(X, y, n_delay)

print("Windowed X shape:", X_td.shape)  # (90001 - n_delay, (n_delay+1)*2)
print("Windowed y shape:", y_td.shape)  # (90001 - n_delay, 2)

X_new_td = make_time_delays_array(X_new.values, n_delay)
print("Windowed X_new shape:", X_new_td.shape)

Windowed X shape: (90000, 2)
Windowed y shape: (90000, 2)
Windowed X_new shape: (30001, 2)


In [3]:
#NN model
model = Sequential([
    Input(shape=((n_delay+1)*2,)),  # Explicitly defining the input shape
    Dense(32, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)  # Output layer with 2 neurons for xI_real and xI_imag
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
history = model.fit(X_td, y_td, epochs=50, validation_split=0.2, verbose=1)

print("Training complete")


# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")


# Predict using the model
y_new_pred = model.predict(X_new_td) 

# Create the DataFrame with predictions
df_predict = pd.DataFrame({
    'xI_predict_real': y_new_pred[:, 0],
    'xI_predict_imag': y_new_pred[:, 1]
})

# Create complex numbers
df_predict['xI_predict'] = (df_predict['xI_predict_real'] + 1j * df_predict['xI_predict_imag']).astype(np.complex128)

# Prepare data for MATLAB file generation
data_to_save = {'xI_predict': df_predict['xI_predict'].values}


output_mat_file_path = 'C:\\Users\\Luke McCubbin\\Box\\NN PA DPD\\RNN_PA_NN\\DohertyTD0.mat'
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f'Predictions saved to {output_mat_file_path}')




Epoch 1/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9451 - loss: 0.1942 - val_accuracy: 0.9794 - val_loss: 0.0119
Epoch 2/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9793 - loss: 0.0119 - val_accuracy: 0.9803 - val_loss: 0.0119
Epoch 3/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9786 - loss: 0.0119 - val_accuracy: 0.9812 - val_loss: 0.0117
Epoch 4/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9789 - loss: 0.0113 - val_accuracy: 0.9802 - val_loss: 0.0112
Epoch 5/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9795 - loss: 0.0115 - val_accuracy: 0.9807 - val_loss: 0.0102
Epoch 6/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9796 - loss: 0.0114 - val_accuracy: 0.9801 - val_loss: 0.0106
Epoch 7/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9788 - loss: 0.0114 - val_accuracy: 0.9807 - val_loss: 0.0116
Epoch 8/50
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9779 - loss: 0.0113 - 